In [1]:
print(123)

123


In [2]:
import sys
import subprocess

print("Python:", sys.executable)

subprocess.run([
    sys.executable, "-m", "pip", "show", "sentence-transformers"
])

Python: /Users/veneraheddergott/LLM_zoomcamp/llm-zoomcamp_2026_vh-1/.venv/bin/python
Name: sentence-transformers
Version: 2.6.1
Summary: Multilingual text embeddings
Home-page: https://www.SBERT.net
Author: Nils Reimers
Author-email: info@nils-reimers.de
License: Apache License 2.0
Location: /Users/veneraheddergott/LLM_zoomcamp/llm-zoomcamp_2026_vh-1/.venv/lib/python3.12/site-packages
Requires: huggingface-hub, numpy, Pillow, scikit-learn, scipy, torch, tqdm, transformers
Required-by: 


CompletedProcess(args=['/Users/veneraheddergott/LLM_zoomcamp/llm-zoomcamp_2026_vh-1/.venv/bin/python', '-m', 'pip', 'show', 'sentence-transformers'], returncode=0)

In [3]:
from sentence_transformers import SentenceTransformer
print("ok")

ok


In [4]:
import numpy as np

In [5]:
from tqdm.auto import tqdm

from ingest import load_faq_data
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

documents = load_faq_data()
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

/Users/veneraheddergott/LLM_zoomcamp/llm-zoomcamp_2026_vh-1/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  0%|          | 0/27 [00:00<?, ?it/s]

In [11]:
import psycopg

conn = psycopg.connect(
    host="127.0.0.1",
    port=5432,
    dbname="faq",
    user="user",
    password="pswd",
    connect_timeout=5
)

print("connected")

connected


In [12]:
conn.execute("""
    DROP TABLE IF EXISTS documents
""")

conn.execute("""
    CREATE TABLE documents (
        id SERIAL PRIMARY KEY,
        course TEXT,
        section TEXT,
        question TEXT,
        answer TEXT,
        embedding vector(384)
    )
""")

<psycopg.Cursor [COMMAND_OK] [INTRANS] (host=127.0.0.1 user=user database=faq) at 0x15681d010>

In [13]:
def vec_to_str(vector):
    return "[" + ",".join(str(x) for x in vector) + "]"

for doc, vec in tqdm(zip(documents, vectors), total=len(documents)):
    conn.execute(
        """
        INSERT INTO documents (course, section, question, answer, embedding)
        VALUES (%s, %s, %s, %s, %s::vector)
        """,
        (doc["course"], doc["section"], doc["question"], doc["answer"],
         vec_to_str(vec))
    )

conn.commit()

  0%|          | 0/1350 [00:00<?, ?it/s]

## Searchuing with cosine similarity"


In [14]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)
query_str = vec_to_str(query_vector)

In [17]:
query_str


'[-0.009474847,-0.069232404,-0.029059552,0.012939028,0.01362286,0.00023433771,-0.07741695,-0.09136969,-0.10466132,-0.01922363,-0.043046,0.03964786,0.0043251924,0.04924715,0.008185845,-0.041844998,-0.04338224,-0.025352692,-0.0013161042,-0.0017762291,-0.0888451,0.044900224,-0.02615121,0.02344962,-0.009180702,0.013769316,0.01856915,0.08787833,-0.032130904,-0.07984384,-0.03690277,0.069717064,0.031200485,0.029062578,0.004983737,0.017343419,0.064096496,-0.05677013,0.0065930216,0.022662383,-0.042738017,-0.027881969,-0.0123385,0.05000446,0.030962799,0.09940239,-0.05988195,-0.0857653,0.019548401,0.030833405,0.025987688,-0.04661563,-0.00039915397,0.01100166,-0.004584875,0.07484971,0.023261916,0.0289103,-0.1124793,-0.0076255873,-0.010046817,-0.04728377,-0.07600154,0.054186597,0.01966643,0.018858774,-0.048078936,-0.014167322,0.12337668,-0.07372962,0.00057702,-0.016402302,0.037018426,0.0066006053,0.099733956,0.016072456,0.068566605,-0.015105575,0.08021407,-0.038274292,-0.024316017,0.081881434,-0.00

In [18]:
#selsct cotúrse and answer
results = conn.execute(
    """
    SELECT course, question, answer,
           1 - (embedding <=> %s::vector) AS similarity
    FROM documents
    ORDER BY embedding <=> %s::vector
    LIMIT 5
    """,
    (query_str, query_str)
).fetchall()

for row in results:
    print(f"[{row[0]}] {row[1]} (similarity: {row[3]:.4f})")

[llm-zoomcamp] I just discovered the course. Can I still join? (similarity: 0.8365)
[machine-learning-zoomcamp] The course has already started. Can I still join it? (similarity: 0.6904)
[mlops-zoomcamp] Course - Can I still join the course after the start date? (similarity: 0.6043)
[data-engineering-zoomcamp] Course: Can I still join the course after the start date? (similarity: 0.5959)
[data-engineering-zoomcamp] Course: Can I get support if I take the course in the self-paced mode? (similarity: 0.5927)


### Filterfing by course

In [19]:
results = conn.execute(
    """
    SELECT course, question, answer,
           1 - (embedding <=> %s::vector) AS similarity
    FROM documents
    WHERE course = %s
    ORDER BY embedding <=> %s::vector
    LIMIT 5
    """,
    (query_str, "llm-zoomcamp", query_str)
).fetchall()

### Create an index for faster search 

In [20]:
conn.execute("""
    CREATE INDEX ON documents
    USING hnsw (embedding vector_cosine_ops)
""")

<psycopg.Cursor [COMMAND_OK] [INTRANS] (host=127.0.0.1 user=user database=faq) at 0x15681c710>

### Wrapping it in a function = kapseln in ein Funktion

In [21]:
def pgvector_search(query, course="llm-zoomcamp", num_results=5):
    query_vector = model.encode(query)
    query_str = vec_to_str(query_vector)
    rows = conn.execute(
        """
        SELECT course, section, question, answer
        FROM documents
        WHERE course = %s
        ORDER BY embedding <=> %s::vector
        LIMIT %s
        """,
        (course, query_str, num_results)
    ).fetchall()

    return [
        {"course": r[0], "section": r[1], "question": r[2], "answer": r[3]}
        for r in rows
    ]

In [22]:
results = pgvector_search("How do I join the course?")

In [24]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Can I run the course locally instead of Codespaces?',
  'answer': 'Yes. Codespaces is just the easiest way for everyone to start with the same environment.\n\nYou can run the course locally if you are comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module.\n\nIf you run locally, make sure you document your setup and keep your environment reproducible.'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question':

In [25]:
from rag_helper import RAGBase

class RAGPgVector(RAGBase):

    def __init__(self, embedder, conn, **kwargs):
        super().__init__(index=None, **kwargs)
        self.embedder = embedder
        self.conn = conn

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        query_str = vec_to_str(query_vector)

        rows = self.conn.execute(
            """
            SELECT course, section, question, answer
            FROM documents
            WHERE course = %s
            ORDER BY embedding <=> %s::vector
            LIMIT %s
            """,
            (self.course, query_str, num_results)
        ).fetchall()

        return [
            {"course": r[0], "section": r[1], "question": r[2], "answer": r[3]}
            for r in rows
        ]

In [27]:
# Initialisiert `llm_client` für Groq (erfordert GROQ_API_KEY)
from dotenv import load_dotenv
from openai import OpenAI
import os
from pathlib import Path

load_dotenv(override=True)

if os.getenv("GROQ_API_KEY") is None:
    raise ValueError("GROQ_API_KEY fehlt. Trage ihn in die .env ein oder setze die Umgebungsvariable vor dem Start.")

llm_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

In [30]:
vector_assistant = RAGPgVector(
    embedder=model,
    conn=conn,
    llm_client=llm_client,
    model="llama-3.3-70b-versatile",
)

In [31]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still sign up for the program even though it has already begun. If you want to receive a certificate, you will need to submit your project while submissions are still being accepted. You can start learning and submitting homework without waiting for a confirmation email, as registration is only used to gauge interest before the start date.'